# CordiaAI — Step 5: ECG Branch (PTB-XL + ViT-B/16)

Downloads PTB-XL (21,837 recordings), converts Lead II to a 224x224 STFT spectrogram, and fine-tunes a pretrained ViT-B/16 to produce the 256-dim ECG embedding E_v, matching the tabular branch's E_t dimensionality for later fusion.

**Before running:** `Runtime -> Change runtime type -> T4 GPU`.

**Note on STFT parameters:** the methodology's `hop=512` figure was written generically. PTB-XL's actual recordings are 10 seconds long; at hop=512 that collapses to only a handful of time frames, too few to resize meaningfully into a 224-wide image. This notebook uses the 100Hz signal (1000 samples/lead — switched from the originally planned 500Hz because PhysioNet throttled the full-dataset zip download to ~230KB/s regardless of connection count; the 100Hz per-file download was ~5x smaller and completed via parallel requests) with `n_fft=64, hop_length=8`, which produces a reasonable spectrogram before the final resize to 224x224 given the shorter signal.

**Note on layer freezing:** this transformers version's ViT uses parameter names like `layers.N....` (plural, with `q_proj`/`k_proj`/`v_proj`) rather than the older `encoder.layer.N` attribute path, so freezing is done via `named_parameters()` string matching rather than direct submodule indexing — see the Model cell.

In [ ]:
!pip install -q wfdb transformers
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (enable GPU in Runtime settings)')

## Download PTB-XL (100Hz records only, ~350MB via parallel per-file requests — see note above on why not the full zip)

In [ ]:
import glob, os
import pandas as pd

PTBXL_ROOT = 'ptbxl_data'

if not glob.glob(f'{PTBXL_ROOT}/ptbxl_database.csv'):
    os.makedirs(PTBXL_ROOT, exist_ok=True)
    !curl -s "https://physionet.org/files/ptb-xl/1.0.3/ptbxl_database.csv?download" -o ptbxl_data/ptbxl_database.csv
    !curl -s "https://physionet.org/files/ptb-xl/1.0.3/scp_statements.csv?download" -o ptbxl_data/scp_statements.csv

    # PhysioNet throttles the full-dataset zip download to ~230KB/s regardless of connection
    # count (tested with 16 parallel aria2c streams -- no improvement, since the cap is
    # per-destination-IP, not per-connection). Downloading the (smaller, 100Hz) per-record
    # files individually in parallel bypasses this entirely: ~20-28 files/sec vs. a ~2 hour
    # projected time for the single big zip.
    df = pd.read_csv('ptbxl_data/ptbxl_database.csv')
    with open('urls.txt', 'w') as f:
        for p in df['filename_lr']:
            f.write(f'https://physionet.org/files/ptb-xl/1.0.3/{p}.dat\n')
            f.write(f'https://physionet.org/files/ptb-xl/1.0.3/{p}.hea\n')

    os.makedirs('ptbxl_data/records100', exist_ok=True)
    !cd ptbxl_data/records100 && cat /content/urls.txt | xargs -P 32 -I XXXX wget -q -x -nH --cut-dirs=4 XXXX

print('PTB-XL root:', PTBXL_ROOT)

## Labels: aggregate SCP codes to the 5 diagnostic superclasses (NORM, MI, STTC, CD, HYP)

In [ ]:
import ast
import pandas as pd

Y = pd.read_csv(f'{PTBXL_ROOT}/ptbxl_database.csv', index_col='ecg_id')
Y['scp_codes'] = Y['scp_codes'].apply(ast.literal_eval)

agg_df = pd.read_csv(f'{PTBXL_ROOT}/scp_statements.csv', index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]

def aggregate_diagnostic(scp_dict):
    classes = {agg_df.loc[code].diagnostic_class for code in scp_dict if code in agg_df.index}
    return list(classes)

Y['diagnostic_superclass'] = Y['scp_codes'].apply(aggregate_diagnostic)

# Keep single-label records only, for a clean 5-class task matching the tabular branch's severity head
Y = Y[Y['diagnostic_superclass'].apply(len) == 1].copy()
Y['label'] = Y['diagnostic_superclass'].apply(lambda x: x[0])

CLASSES = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
Y = Y[Y['label'].isin(CLASSES)].copy()
Y['label_idx'] = Y['label'].map({c: i for i, c in enumerate(CLASSES)})

print(f'{len(Y)} single-label records kept (of {len(pd.read_csv(f"{PTBXL_ROOT}/ptbxl_database.csv"))} total)')
print(Y['label'].value_counts())

## Precompute spectrograms (once, cached in memory as uint8 to fit comfortably in RAM)
~150KB/image x ~19-20k records ≈ 3GB. Progress prints every 2000 records so this is easy to monitor.

In [ ]:
import numpy as np
import librosa
import wfdb
from PIL import Image

LEAD_II_IDX = 1  # PTB-XL lead order: I, II, III, AVR, AVL, AVF, V1-V6

def ecg_to_spectrogram(record_path: str) -> np.ndarray:
    signal, _ = wfdb.rdsamp(record_path)
    lead = signal[:, LEAD_II_IDX].astype(np.float32)

    stft = librosa.stft(lead, n_fft=64, hop_length=8, window='hann')
    magnitude_db = librosa.amplitude_to_db(np.abs(stft), ref=np.max)

    normalised = (magnitude_db - magnitude_db.min()) / (magnitude_db.max() - magnitude_db.min() + 1e-8)
    img = Image.fromarray((normalised * 255).astype(np.uint8)).resize((224, 224), Image.BILINEAR)
    return np.stack([np.array(img)] * 3, axis=-1)  # replicate to 3 channels

records = Y['filename_lr'].tolist()  # 100Hz (switched from 500Hz: full-zip download was throttled by PhysioNet)
labels = Y['label_idx'].tolist()

images = np.zeros((len(records), 224, 224, 3), dtype=np.uint8)
valid_mask = np.ones(len(records), dtype=bool)

for i, rel_path in enumerate(records):
    try:
        images[i] = ecg_to_spectrogram(f'{PTBXL_ROOT}/{rel_path}')
    except Exception as e:
        valid_mask[i] = False
    if (i + 1) % 2000 == 0:
        print(f'{i + 1}/{len(records)} processed')

images = images[valid_mask]
labels = np.array(labels)[valid_mask]
strat_fold = Y['strat_fold'].values[valid_mask]
print(f'Done. {len(images)} usable spectrograms ({(~valid_mask).sum()} records failed to load and were dropped).')

## Split (PTB-XL's official recommended protocol: folds 1-8 train, 9 val, 10 test)

In [ ]:
train_mask = strat_fold <= 8
val_mask = strat_fold == 9
test_mask = strat_fold == 10

print(f'Train: {train_mask.sum()}  Val: {val_mask.sum()}  Test: {test_mask.sum()}')

## Model: ViT-B/16, freeze layers 1-8, fine-tune layers 9-12 + a 256-dim projection head

In [ ]:
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import ViTModel

class SpectrogramDataset(Dataset):
    def __init__(self, images, labels):
        self.images = images
        self.labels = labels

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = self.images[idx].astype(np.float32) / 255.0
        img = (img - 0.5) / 0.5  # matches google/vit-base-patch16-224-in21k's preprocessor config
        img = torch.tensor(img).permute(2, 0, 1)  # HWC -> CHW
        return img, self.labels[idx]

class ECGViT(nn.Module):
    def __init__(self, n_classes=5, d_embed=256, freeze_below_layer=8, total_layers=12):
        super().__init__()
        self.vit = ViTModel.from_pretrained('google/vit-base-patch16-224-in21k')
        # Freeze/unfreeze by parameter name rather than by indexing self.vit.encoder.layer
        # directly: this transformers version names ViT parameters "layers.N...." (plural,
        # with q_proj/k_proj/v_proj) rather than the older "encoder.layer.N" attribute path,
        # and named_parameters() is robust to that kind of internal naming difference.
        for name, p in self.vit.named_parameters():
            p.requires_grad = any(f'layers.{i}.' in name for i in range(freeze_below_layer, total_layers))

        self.project = nn.Sequential(nn.Linear(768, d_embed), nn.LayerNorm(d_embed))
        self.classifier = nn.Linear(d_embed, n_classes)

    def embed(self, x):
        cls_token = self.vit(pixel_values=x).last_hidden_state[:, 0]
        return self.project(cls_token)

    def forward(self, x):
        return self.classifier(self.embed(x))

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ECGViT().to(device)
print(f'Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

## Train

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, f1_score

EPOCHS = 5
BATCH_SIZE = 32

train_loader = DataLoader(SpectrogramDataset(images[train_mask], labels[train_mask]),
                           batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(SpectrogramDataset(images[val_mask], labels[val_mask]), batch_size=64)
test_loader = DataLoader(SpectrogramDataset(images[test_mask], labels[test_mask]), batch_size=64)

optimiser = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=2e-5, weight_decay=1e-4)

# Class-weighted loss: HYP has only 415/12956 training examples (3.2%) vs NORM's 7243 (56%).
# Unweighted training scored 0.00 precision/recall/F1 on HYP -- the model never learned it.
# Weight = inverse frequency, normalized so weights average to 1 (keeps the overall loss
# magnitude comparable to the unweighted run, so lr=2e-5 is still a sensible starting point).
class_counts = np.bincount(labels[train_mask], minlength=len(CLASSES))
class_weights = torch.tensor(len(labels[train_mask]) / (len(CLASSES) * class_counts), dtype=torch.float32, device=device)
print('Class counts:', dict(zip(CLASSES, class_counts)))
print('Class weights:', dict(zip(CLASSES, class_weights.cpu().numpy().round(2))))

criterion = nn.CrossEntropyLoss(weight=class_weights)

def run_eval(loader):
    model.eval()
    preds, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            preds.extend(model(xb).argmax(dim=1).cpu().numpy())
            targets.extend(yb.numpy())
    return np.array(preds), np.array(targets)

best_val_f1, best_state = -1.0, None
for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimiser.zero_grad()
        loss = criterion(model(xb), yb)
        loss.backward()
        optimiser.step()
        total_loss += loss.item() * len(xb)

    val_preds, val_targets = run_eval(val_loader)
    val_f1 = f1_score(val_targets, val_preds, average='macro')
    val_acc = accuracy_score(val_targets, val_preds)
    print(f'Epoch {epoch}  train_loss={total_loss / train_mask.sum():.4f}  val_acc={val_acc:.4f}  val_f1={val_f1:.4f}')

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print(f'\nBest validation macro F1: {best_val_f1:.4f}')

## Evaluate on held-out test fold (fold 10, never used for training or model selection)

In [ ]:
test_preds, test_targets = run_eval(test_loader)
print(f"Test accuracy: {accuracy_score(test_targets, test_preds):.4f}")
print(f"Test macro F1: {f1_score(test_targets, test_preds, average='macro'):.4f}\n")
print(classification_report(test_targets, test_preds, target_names=CLASSES, zero_division=0))

## Save weights + extract E_v embeddings for later fusion, then download

In [ ]:
torch.save(best_state, 'ecg_vit.pt')

def extract_embeddings(loader):
    model.eval()
    embeds, targets = [], []
    with torch.no_grad():
        for xb, yb in loader:
            embeds.append(model.embed(xb.to(device)).cpu().numpy())
            targets.extend(yb.numpy())
    return np.concatenate(embeds), np.array(targets)

for split_name, loader in [('train', train_loader), ('val', val_loader), ('test', test_loader)]:
    emb, tgt = extract_embeddings(DataLoader(loader.dataset, batch_size=64))
    cols = [f'embed_{i}' for i in range(emb.shape[1])]
    df = pd.DataFrame(emb, columns=cols)
    df['label'] = tgt
    df.to_csv(f'ecg_embeddings_{split_name}.csv', index=False)
    print(f'Saved ecg_embeddings_{split_name}.csv {df.shape}')

from google.colab import files
files.download('ecg_vit.pt')
files.download('ecg_embeddings_train.csv')
files.download('ecg_embeddings_val.csv')
files.download('ecg_embeddings_test.csv')